# Predict customer orders with synthetic TPC-H tables on Snowflake

How many orders will a recently active customer place in the next 90 days? Train Kumo Engine on synthetic customer–order–line-item–part history, then score newly added customers. Binary classification and order-to-part ranking are optional alternatives.

Use a Snowflake GPU notebook with supported local SSD scratch, PyPI access and the [Snowflake integration](https://docs.nvidia.com/sdgm/kumo-engine/get-started-with-kumo-engine/installation/snowflake-notebook). The warehouse reads/writes tables; notebook GPU compute trains the model. Allow room for the materialized graph and retained artifacts. Synthetic metrics are demonstration only.


## Enter your configuration

Choose existing source and output schemas that you own. Training publishes the model and scoring writes a unique output table. The scoring date follows the generated 2023–2025 history. Rerun configuration after restarting the kernel.

In [ ]:
from uuid import uuid4


from datetime import datetime

WAREHOUSE_NAME = "<YOUR_WAREHOUSE>"
DATABASE_NAME = "<YOUR_DATABASE>"  # Use your own existing database.
SCHEMA_NAME = "<YOUR_SCHEMA>"  # Load generated tables here.

MODEL_NAME = "<YOUR_MODEL_NAME>"  # Model registry name; each fit adds a version.
PREDICTION_TIME = datetime(2026, 1, 5)  # Generated history spans 2023 through 2025.
OUTPUT_SCHEMA = "<YOUR_OUTPUT_SCHEMA>"  # Use your own existing database.
PREDICTION_OUTPUT_TABLE = ("<YOUR_PREDICTION_TABLE>") + "_" + uuid4().hex[:8]
# Use a schema containing only your generated example tables.


## Install Kumo Engine from PyPI

Run this installation cell once. **Then restart the kernel, rerun configuration, skip installation, and continue at imports.** Do not use Run All across the restart boundary. Installation uses the active interpreter.

In [ ]:
%pip install kumo-engine[snowflake]

<a id="runtime"></a>
## Import and configure scratch

After each kernel restart, run configuration and these cells. They do not depend on variables created during installation. Configure scratch before constructing any Kumo table, graph, or trainer.

In [ ]:
import shutil
import tempfile
from pathlib import Path

import kumo.engine as kumo
from kumo.engine import SnowflakeConnector
from kumoapi.graph import Edge
from kumoapi.jobs import PredictionArtifactType, WriteMode

In [ ]:
SCRATCH_DIR = "/tmp/kumoai-local"
scratch = Path(SCRATCH_DIR)
scratch.mkdir(parents=True, exist_ok=True)
with tempfile.TemporaryFile(dir=scratch) as probe:
    probe.write(b"kumo scratch check")
    probe.flush()
kumo.configure(scratch_dir=SCRATCH_DIR)
print("Kumo version:", kumo.__version__)
print("Free working space (GiB):", round(shutil.disk_usage(scratch).free / 2**30, 1))

The printed version identifies the installed build. The probe checks write access and free space at this path; it does not establish that the path is on supported local storage. Confirm the backing disk and capacity before materializing data.

## Generate synthetic example data

Run the next cell to create initial history and reserve new customers with their observed orders. You can skim the generation code; the graph workflow starts below. Load only `initial/` files now, using uppercase table and column names. Reserved rows are appended after fitting; the generator creates local files only.


In [ ]:
from pathlib import Path
import tempfile
import numpy as np
import pandas as pd

def make_tpch(seed, customer_start, customer_count, order_start, rows, start, days):
    rng = np.random.default_rng(seed)
    customer_ids = np.arange(customer_start, customer_start + customer_count)
    part_ids = np.arange(80)
    customer = rng.choice(customer_ids, rows)
    day = rng.integers(0, days, rows)
    keep = (customer % 4 != 0) | (day < int(days * 0.6))
    customer, day = customer[keep], day[keep]
    item = customer % 4 * 20 + rng.integers(20, size=len(day))
    dates = pd.Timestamp(start) + pd.to_timedelta(day, unit="D")
    amount = np.round(10 + item * 1.5 + rng.normal(0, 2, len(day)), 2)
    ids = np.arange(order_start, order_start + len(day))
    return {
        "customer": pd.DataFrame(dict(c_custkey=customer_ids, c_name=[f"Synthetic customer {i}" for i in customer_ids], c_nationkey=customer_ids % 5, c_mktsegment=customer_ids % 4, c_acctbal=100 + customer_ids)),
        "part": pd.DataFrame(dict(p_partkey=part_ids, p_name=[f"Synthetic part {i}" for i in part_ids], p_brand=part_ids % 5, p_size=1 + part_ids % 10, p_container="BOX", p_retailprice=20 + part_ids * 2, p_type=part_ids // 20)),
        "orders": pd.DataFrame(dict(o_orderkey=ids, o_custkey=customer, o_orderdate=dates, o_totalprice=amount, o_shippriority=customer % 3, o_orderstatus=np.where(customer % 2, "F", "O"), o_orderpriority=np.where(customer % 2, "HIGH", "LOW"))),
        "lineitem": pd.DataFrame(dict(lineitem_id=ids, l_shipmode="TRUCK", l_orderkey=ids, l_partkey=item, l_suppkey=item % 10, l_linenumber=1, l_quantity=1 + customer % 4, l_extendedprice=amount, l_discount=0.05, l_tax=0.02, l_shipdate=dates + pd.Timedelta(days=2), l_commitdate=dates + pd.Timedelta(days=1), l_receiptdate=dates + pd.Timedelta(days=3), l_returnflag="N", l_linestatus="O")),
    }

frames = make_tpch(42, 0, 200, 0, 5000, "2023-01-01", 1095)
reserved = make_tpch(43, 200, 20, 1_000_000, 600,
                     pd.Timestamp(PREDICTION_TIME) - pd.Timedelta(days=180), 177)
reserved.pop("part")  # New customers reuse the original catalog.
assert set(reserved["customer"].c_custkey).isdisjoint(frames["customer"].c_custkey)
assert set(reserved["orders"].o_orderkey).isdisjoint(frames["orders"].o_orderkey)
assert reserved["orders"].o_custkey.isin(reserved["customer"].c_custkey).all()
assert set(reserved["orders"].o_custkey) == set(reserved["customer"].c_custkey)
assert reserved["lineitem"].l_orderkey.isin(reserved["orders"].o_orderkey).all()
assert reserved["lineitem"].l_partkey.isin(frames["part"].p_partkey).all()
assert reserved["lineitem"].l_shipdate.lt(pd.Timestamp(PREDICTION_TIME)).all()
frames = {name.upper(): frame.rename(columns=str.upper) for name, frame in frames.items()}
reserved = {name.upper(): frame.rename(columns=str.upper) for name, frame in reserved.items()}
DATA_DIR = Path(tempfile.mkdtemp(prefix="synthetic-tpch-"))
initial_dir = DATA_DIR / "initial"
initial_dir.mkdir()
for name, frame in frames.items():
    frame.to_parquet(initial_dir / f"{name}.parquet", index=False)
print("Load only these initial files before training:", initial_dir)
print({name: len(frame) for name, frame in frames.items()})


## Connect to your loaded tables

Manually load each `initial/` Parquet file into the configured source schema under its filename stem. Preserve numeric keys and date columns. The Snowflake connector uses this notebook’s runtime OAuth identity.


In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()


In [ ]:
connector = kumo.SnowflakeConnector.from_spcs_env(
    warehouse=WAREHOUSE_NAME, database=DATABASE_NAME, schema_name=SCHEMA_NAME,
)
print("Tables:", connector.table_names())

## Describe the four tables

A primary key identifies a row. A physical type describes its stored representation, such as a date; a semantic type describes its modeling role, such as a category or numerical value.

In [ ]:
customer = (
    kumo.Table.from_source_table(
        connector["CUSTOMER"],
        ["C_CUSTKEY", "C_NATIONKEY", "C_MKTSEGMENT", "C_ACCTBAL"],
        primary_key="C_CUSTKEY",
    )
)

part = (
    kumo.Table.from_source_table(
        connector["PART"],
        ["P_PARTKEY", "P_NAME", "P_BRAND", "P_TYPE",
         "P_SIZE", "P_CONTAINER", "P_RETAILPRICE"],
        primary_key="P_PARTKEY",
    )
)

Event timestamps let Kumo separate information available before a prediction boundary from future outcomes. Using future information as an input causes temporal leakage.

In [ ]:
orders = (
    kumo.Table.from_source_table(
        connector["ORDERS"],
        ["O_ORDERKEY", "O_CUSTKEY", "O_ORDERDATE", "O_ORDERSTATUS",
         "O_ORDERPRIORITY", "O_TOTALPRICE"],
        primary_key="O_ORDERKEY",
        time_column="O_ORDERDATE",
    )
)

In [ ]:
lineitem = (
    kumo.Table.from_source_table(
        connector["LINEITEM"],
        ["LINEITEM_ID", "L_ORDERKEY", "L_PARTKEY", "L_SHIPDATE",
         "L_QUANTITY", "L_EXTENDEDPRICE", "L_DISCOUNT"],
        primary_key="LINEITEM_ID",
        time_column="L_SHIPDATE",
    )
)

## Connect the tables and materialize a snapshot

A **graph** describes the tables and their relationships. Each arrow below goes from a foreign-key column to the destination table's primary key:

```text
LINEITEM.L_PARTKEY  -> PART.P_PARTKEY
LINEITEM.L_ORDERKEY -> ORDERS.O_ORDERKEY
ORDERS.O_CUSTKEY    -> CUSTOMER.C_CUSTKEY
```

Validation checks the graph definition; it is not a blanket guarantee of source-data quality. A **snapshot** materializes source data into the local graph index used for training and prediction. Snapshot creation reads the tables and consumes local disk. Reuse requires an unchanged, valid retained snapshot; loading a model does not restore a lost index.

In [ ]:

graph = kumo.Graph(
    tables=[customer, part, orders, lineitem],
    edges=[
        Edge(src_table="ORDERS", fkey="O_CUSTKEY", dst_table="CUSTOMER"),
        Edge(src_table="LINEITEM", fkey="L_ORDERKEY", dst_table="ORDERS"),
        Edge(src_table="LINEITEM", fkey="L_PARTKEY", dst_table="PART"),
    ],
)

graph.validate()

## Inspect the graph

Inspect the actual graph’s tables and relationships before materialization. The [graph visualization](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/set-up-a-graph/graphs) call is a commented preview because it is unavailable in the supported API.

In [ ]:
# graph.visualize(show_cols=True)

### Materialize the graph

Read the initial loaded tables into the training snapshot.


In [ ]:
snapshot = graph.snapshot()

print("Snapshot:", snapshot.id)

## Define the prediction question

Predictive Query Language (PQL) is Kumo Engine's declarative syntax for defining a predictive modeling task: it specifies the target, the entity to score, and the data context. This default query predicts how many orders a customer active in the past year will place in the next 90 days.

In [ ]:
# Regression: predict how many orders a customer active in the past year will place in the next 90 days.
pquery = kumo.PredictiveQuery(
    graph=graph,
    query=(
        "PREDICT COUNT(ORDERS.*, 0, 90, days) "
        "FOR EACH CUSTOMER.C_CUSTKEY "
        "WHERE COUNT(ORDERS.*, -365, 0, days) > 0"
    ),
)
pquery.validate()
print("Task type:", pquery.get_task_type())

### Optional task alternatives

The next two code cells are commented out. To try one, skip or comment out the default regression query cell above, uncomment exactly one alternative, and run it before training. 

In [ ]:
# Binary classification: predict whether a customer active in the past year will order again in the next 90 days.
# pquery = kumo.PredictiveQuery(
#     graph=graph,
#     query="""
#         PREDICT COUNT(ORDERS.*, 0, 90, days) > 0
#         FOR EACH CUSTOMER.C_CUSTKEY
#         WHERE COUNT(ORDERS.*, -365, 0, days) > 0
#     """,
# )

# pquery.validate()
# print("Task type:", pquery.get_task_type())

In [ ]:
# Link prediction: rank up to five parts that may ship in the next 90 days for an order with recent line items.
# pquery = kumo.PredictiveQuery(
#     graph=graph,
#     query=(
#         "PREDICT LIST_DISTINCT(LINEITEM.L_PARTKEY, 0, 90, days) "
#         "RANK TOP 5 FOR EACH ORDERS.O_ORDERKEY "
#         "WHERE COUNT(LINEITEM.*, -365, 0, days) > 0"
#     ),
# )
# pquery.validate()
# print("Task type:", pquery.get_task_type())

### Generate training rows

Use approximate chronological 70/15/15 splits so validation and test follow training. Anchors are 90 days apart over the most recent 900 days; the generation plan leaves the target window observable. Inspect actual split dates and row counts (`0` train, `1` validation, `2` test).

In [ ]:
from kumoapi.model_plan import TrainingTableGenerationPlan
from kumo.engine.trainer.output_config import TrainingTableOutputConfig

generation_plan = TrainingTableGenerationPlan(
    split="ApproxDateOffsetSplit([0.7, 0.15, 0.15])",
    train_start_offset=900, train_end_offset=0, timeframe_step=90,
)
train_table = pquery.generate_training_table(snapshot, plan=generation_plan)
local_artifacts = DATA_DIR / "artifacts"
local_artifacts.mkdir(exist_ok=True)
local_output = kumo.FileConnector(root_dir=str(local_artifacts))
train_table.export(TrainingTableOutputConfig(connector=local_output, table_name="training_rows/"))
training_rows = pd.read_parquet(local_artifacts / "training_rows")
print(training_rows.groupby("SPLIT").agg(rows=("ENTITY1", "size"), first_anchor=("TIMESTAMP", "min"), last_anchor=("TIMESTAMP", "max")))

## Configure the model

The suggested model plan controls architecture, sampling, and optimization. The model plan is fully customizable; inspect and adjust its settings and validation selection metric for your task before creating `trainer = kumo.Trainer(model_plan)`. Compare like-for-like validation metrics and reserve test metrics for evaluating the selected model.

In [ ]:
from kumoapi.model_plan import RunMode

model_plan = pquery.suggest_model_plan(snapshot, run_mode=RunMode.FAST)

model_plan.training_job.refit_trainval = False
model_plan.training_job.refit_full = False
trainer = kumo.Trainer(model_plan)

### Train the model

After creating a Trainer from the model plan, call `fit()` with the training table and compatible graph snapshot. The fit step also publishes the model to Snowflake.

For a deliberately local-only experiment, set `model_output = None` in this cell before running it.

In [ ]:
model_output = kumo.ModelOutputConfig(
    connector, location=kumo.ModelRegistryLocation(MODEL_NAME),
)

training_result = trainer.fit(
    train_table, snapshot=snapshot, export_config=model_output,
)

### Evaluate

**Synthetic metrics — demonstration only.** For the default regression task, MAE/RMSE measure order-count error; lower is better. For binary classification, AUROC measures repeat-order ranking. For the link alternative, ranking metrics reward relevant parts near the top. Inspect each metric’s validation/test stage; use validation for choices and reserve test for assessment. Retain the complete model bundle before the runtime ends.

In [ ]:
print("Synthetic metrics — demonstration only:", training_result.metrics)
print("Reusable bundle:", training_result.bundle_dir)

## Batch predictions

Append the reserved new customers, orders and line items to the same source tables after training. The shared part catalog stays unchanged. All reserved events precede the scoring anchor. The original training snapshot remains available; a fresh connector and snapshot read the updated sources.

In [ ]:
append_dir = DATA_DIR / "append"
append_dir.mkdir(exist_ok=False)
for name, frame in reserved.items():
    frame.to_parquet(append_dir / f"{name}.parquet", index=False)
print("Append these files once to the matching source tables:", append_dir)
print({name: len(frame) for name, frame in reserved.items()})

### Load and verify the appended rows

Manually append each `append/` file to its matching source table, preserving existing rows. Append exactly once; do not reload `initial/`, replace tables, or append the part catalog. Continue only after the upload completes. These checks compare the loaded data with the expected original-plus-new tables.


In [ ]:
# Verify that manual appends added exactly the reserved rows, with unique keys.
keys = {'CUSTOMER': 'C_CUSTKEY', 'PART': 'P_PARTKEY', 'ORDERS': 'O_ORDERKEY', 'LINEITEM': 'LINEITEM_ID'}
for name, initial in frames.items():
    loaded = session.table([DATABASE_NAME, SCHEMA_NAME, name]).to_pandas()
    expected = pd.concat([initial, reserved[name]], ignore_index=True) if name in reserved else initial
    key = keys[name]
    assert loaded[key].is_unique, f"Duplicate keys in {name}; do not append twice."
    assert len(loaded) == len(expected), f"Complete the one-time append for {name}."
    assert set(loaded[key]) == set(expected[key]), f"Unexpected IDs in {name}."
    pd.testing.assert_frame_equal(
        loaded[expected.columns].sort_values(key).reset_index(drop=True),
        expected.sort_values(key).reset_index(drop=True),
        check_dtype=False, check_exact=False,
    )


### Prepare the new-record population

Rebuild the same graph definitions through a fresh connector, then prepare only the reserved entity IDs with their scoring anchor. No future outcomes are supplied.


In [ ]:
from dataclasses import replace

scoring_connector = kumo.SnowflakeConnector.from_spcs_env(
    warehouse=WAREHOUSE_NAME, database=DATABASE_NAME, schema_name=SCHEMA_NAME)
scoring_graph = kumo.Graph(
    tables=[replace(table, connector=scoring_connector) for table in graph.tables],
    edges=graph.edges,
)
scoring_graph.validate()
scoring_snapshot = scoring_graph.snapshot()
assert scoring_snapshot.id != snapshot.id
scoring_query = kumo.PredictiveQuery(scoring_graph, pquery.query)
scoring_query.validate()
new_ids = (reserved["ORDERS"]["O_ORDERKEY"] if pquery.get_task_type().is_link_pred
           else reserved["CUSTOMER"]["C_CUSTKEY"])
scoring_population = pd.DataFrame({"ENTITY1": new_ids, "TIMESTAMP": PREDICTION_TIME})
assert len(scoring_population) >= 2 and scoring_population.ENTITY1.is_unique
assert not any(name.startswith("TARGET") for name in scoring_population)
scoring_population.to_parquet(local_artifacts / "scoring_population.parquet", index=False)
prediction_table = kumo.PredictionTable.from_source_table(
    local_output["scoring_population"], pq=scoring_query, snapshot=scoring_snapshot,
    plan=trainer.training_table_generation_plan,
)
population = pd.read_parquet(prediction_table.table_path)
assert len(population) == len(scoring_population)
assert set(population.ENTITY1) == set(new_ids)
print(population.head(8).to_string(index=False))


### Configure prediction output

Default regression and binary classification score only the new customers. The optional link query instead scores only the new orders and returns five part candidates per order. Use the unique output destination configured for this run.

In [ ]:
from kumoapi.task import TaskType

# The output schema must exist before Kumo writes to it.


# Separate connector for writes, reusing the source connector's identity and warehouse.
output_connector = SnowflakeConnector(
    identity=connector.identity,
    warehouse=connector.warehouse,
    database=connector.database,
    role=connector.role,
    schema_name=OUTPUT_SCHEMA,
)

# Specific threshold or candidate count are required for binary classification and link prediction tasks respectively. 
prediction_options = {}
task_type = pquery.get_task_type()
if task_type == TaskType.BINARY_CLASSIFICATION:
    prediction_options["binary_classification_threshold"] = 0.5
elif task_type.is_link_pred:
    prediction_options["num_classes_to_return"] = 5

prediction_output = kumo.PredictionOutputConfig(
    connector=output_connector,
    artifact_type=PredictionArtifactType.PREDICTIONS,
    table_name=PREDICTION_OUTPUT_TABLE,
    write_mode=WriteMode.OVERWRITE,
)

### Score the new records

Apply the fitted model to the updated snapshot without retraining.


In [ ]:
prediction_result = trainer.predict(
    prediction_table, snapshot=scoring_snapshot, output_config=prediction_output, **prediction_options,
)


### Inspect and save results

Verify IDs against the reserved scoring population. Regression returns one order-count prediction per customer; binary classification returns its probability and decision. Link prediction returns five ranked part rows per order. Save the full small batch and inspect ten rows.

In [ ]:
import pandas as pd
from pathlib import Path
from uuid import uuid4

population = pd.read_parquet(prediction_table.table_path)
assert len(population) >= 2, "Choose at least two scoring rows."
scoring_rows = population[[name for name in ("ENTITY1", "TIMESTAMP") if name in population]]
assert not any(str(column).startswith("TARGET") for column in scoring_rows)
batch_scores = prediction_result.frame
assert set(batch_scores["ENTITY"]) == set(population["ENTITY1"])
if pquery.get_task_type().is_link_pred:
    assert batch_scores.groupby("ENTITY").size().eq(5).all()
    assert not batch_scores.duplicated(["ENTITY", "CLASS"]).any()
else:
    assert len(batch_scores) == len(population)
    assert batch_scores.ENTITY.is_unique
print(batch_scores.head(10).to_string(index=False))
saved_predictions = Path.cwd() / ("batch-predictions-" + uuid4().hex + ".parquet")
batch_scores.to_parquet(saved_predictions, index=False)
print("Saved:", saved_predictions)

## Retain and reuse

Retain the complete model bundle and prediction export in durable storage. Rebuild the compatible source graph and scoring snapshot in a later session, then load the bundle with `kumo.Trainer.load()`; the model does not contain source data. See [model reuse](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/train-a-model/publish-and-retrieve-models).
